In [2]:
import re

import pandas as pd
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

import torch
import numpy as np
import evaluate

from torch.utils.data import DataLoader, TensorDataset

C:\Natural Language Processing\Entity Sentiment Analysis\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
MODEL_NAME = "cardiffnlp/twitter-roberta-base-2022-154m"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at cardiffnlp/twitter-roberta-base-2022-154m and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
df_train = pd.read_parquet("../../datasets/tweet_eval/train-00000-of-00001.parquet")
df_val = pd.read_parquet("../../datasets/tweet_eval/validation-00000-of-00001.parquet")
df_test = pd.read_parquet("../../datasets/tweet_eval/test-00000-of-00001.parquet")


In [3]:
df_train.head()

,text,label
0,"""QT @user In the original draft of the 7th boo...",2
1,"""Ben Smith / Smith (concussion) remains out of...",1
2,Sorry bout the stream last night I crashed out...,1
3,Chase Headley's RBI double in the 8th inning o...,1
4,@user Alciato: Bee will invest 150 million in ...,2


In [4]:
df_val.head()

,text,label
0,Dark Souls 3 April Launch Date Confirmed With ...,1
1,"""National hot dog day, national tequila day, t...",2
2,When girls become bandwagon fans of the Packer...,0
3,@user I may or may not have searched it up on ...,1
4,Here's your starting TUESDAY MORNING Line up a...,1


In [5]:
df_test.head()

,text,label
0,@user @user what do these '1/2 naked pics' hav...,1
1,OH: “I had a blue penis while I was this” [pla...,1
2,"@user @user That's coming, but I think the vic...",1
3,I think I may be finally in with the in crowd ...,2
4,"@user Wow,first Hugo Chavez and now Fidel Cast...",0


In [6]:
def preprocess_tweet(text):
    # Replace URLs
    text = re.sub(r"http\S+|www\.\S+", "HTTPURL", text)
    # Replace @mentions
    text = re.sub(r"@\w+", "@USER", text)
    return text

In [7]:
df_train["text"] = df_train["text"].apply(preprocess_tweet)
df_val["text"] = df_val["text"].apply(preprocess_tweet)
df_test["text"] = df_test["text"].apply(preprocess_tweet)

In [8]:
train_dataset = Dataset.from_pandas(df_train)
val_dataset = Dataset.from_pandas(df_val)
test_dataset = Dataset.from_pandas(df_test)


def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

In [9]:
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 12284/12284 [00:00<00:00, 14295.07 examples/s]


In [12]:
print(tokenized_train)
print(tokenized_val)
print(tokenized_test)

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 45615
})
Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 2000
})
Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 12284
})


In [14]:
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # logits itu kayak hasil tebakan murni model berupa angka
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [18]:
MODEL_SAVE_PATH = "../../models/roberta-sentiment-finetuned-154m"

In [19]:
training_args = TrainingArguments(
    output_dir=MODEL_SAVE_PATH,
    eval_strategy="steps",         # eval lebih sering, bukan cuma tiap epoch
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,            # simpan max 2 checkpoint, hemat storage
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,              # ~1700 steps warmup, bantu epoch 1
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,        # tambahkan ini — kasih tau "accuracy makin tinggi makin bagus"
)

In [20]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

In [21]:
print("Start training")
trainer.train()

Start training


Step,Training Loss,Validation Loss,Accuracy
500,0.496700,0.673412,0.709500
1000,0.658200,0.740977,0.683500
1500,0.672900,0.628765,0.706500
2000,0.668300,0.668343,0.684000
2500,0.658400,0.615067,0.728000
3000,0.650500,0.678440,0.719500
3500,0.629800,0.660670,0.720000
4000,0.652200,0.590545,0.737000
4500,0.610900,0.586404,0.753500
5000,0.605400,0.608396,0.746000


TrainOutput(global_step=17106, training_loss=0.49434149447873393, metrics={'train_runtime': 4284.263, 'train_samples_per_second': 31.941, 'train_steps_per_second': 3.993, 'total_flos': 3.600575564898816e+16, 'train_loss': 0.49434149447873393, 'epoch': 3.0})

In [22]:
trainer.save_model(MODEL_SAVE_PATH)
tokenizer.save_pretrained(MODEL_SAVE_PATH) 

('../../models/bert-sentiment-finetuned-154m\\tokenizer_config.json',
 '../../models/bert-sentiment-finetuned-154m\\special_tokens_map.json',
 '../../models/bert-sentiment-finetuned-154m\\vocab.json',
 '../../models/bert-sentiment-finetuned-154m\\merges.txt',
 '../../models/bert-sentiment-finetuned-154m\\added_tokens.json',
 '../../models/bert-sentiment-finetuned-154m\\tokenizer.json')

In [30]:
MODEL_PATH = "../../models/roberta-sentiment-finetuned-154m"

# 1. load model
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

# 2. metrics
metric_accuracy = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = metric_accuracy.compute(predictions=predictions, references=labels)
    f1 = metric_f1.compute(predictions=predictions, references=labels, average="macro")
    return {"accuracy": accuracy["accuracy"], "f1": f1["f1"]}

# 3. evaluate
trainer = Trainer(model=model, compute_metrics=compute_metrics)
results = trainer.evaluate(eval_dataset=tokenized_val)
print(results)

{'eval_loss': 0.6522096991539001, 'eval_model_preparation_time': 0.001, 'eval_accuracy': 0.7565, 'eval_f1': 0.7394346665955794, 'eval_runtime': 11.5274, 'eval_samples_per_second': 173.5, 'eval_steps_per_second': 21.687}


In [10]:
MODEL_PATH = "../../models/roberta-sentiment-finetuned-154m"

# 1. load model
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

# 2. metrics
metric_accuracy = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = metric_accuracy.compute(predictions=predictions, references=labels)
    f1 = metric_f1.compute(predictions=predictions, references=labels, average="macro")
    return {"accuracy": accuracy["accuracy"], "f1": f1["f1"]}

# 3. evaluate
trainer = Trainer(model=model, compute_metrics=compute_metrics)
results = trainer.evaluate(eval_dataset=tokenized_test)
print(results)

{'eval_loss': 0.713803768157959, 'eval_model_preparation_time': 0.002, 'eval_accuracy': 0.709296646043634, 'eval_f1': 0.7111621156306418, 'eval_runtime': 220.6307, 'eval_samples_per_second': 55.677, 'eval_steps_per_second': 6.962}
